In [0]:
%python
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
-- fk_analysts: USE CATALOG; USE SCHEMA and SELECT on gold only
GRANT USE CATALOG ON CATALOG fk_1368 TO `fk_analysts`;
GRANT USE SCHEMA, SELECT ON SCHEMA fk_1368.gold TO `fk_analysts`;

-- fk_engineers: USE CATALOG; USE SCHEMA, SELECT, MODIFY on bronze/silver/gold/ops;
--               READ VOLUME, WRITE VOLUME on landing.raw
GRANT USE CATALOG ON CATALOG fk_1368 TO `fk_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA fk_1368.bronze TO `fk_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA fk_1368.silver TO `fk_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA fk_1368.gold TO `fk_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA fk_1368.ops TO `fk_engineers`;
GRANT READ VOLUME, WRITE VOLUME ON VOLUME fk_1368.landing.raw TO `fk_engineers`;

In [0]:
-- G2: Entitlement table (1 mark)
-- Create ops.user_entitlements for PII and row-level access decisions
-- allowed_city holds a city name or ALL

CREATE TABLE IF NOT EXISTS fk_1368.ops.user_entitlements (
  user_email    STRING,
  allowed_city  STRING,
  can_view_pii  BOOLEAN
);

-- Seed a row for the current user so they can see unmasked data and all cities
MERGE INTO fk_1368.ops.user_entitlements t
USING (
  SELECT current_user() AS user_email, 'ALL' AS allowed_city, true AS can_view_pii
) s
ON t.user_email = s.user_email
WHEN MATCHED AND (
  t.allowed_city IS DISTINCT FROM s.allowed_city
  OR t.can_view_pii IS DISTINCT FROM s.can_view_pii
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
-- G3: Column masks (2 marks)
-- Mask email and phone in gold.dim_customer
-- Admins (fk_admins) or users with can_view_pii=TRUE see real values
-- Everyone else sees masked values

-- Email mask: first char + ***@ + domain (e.g., rahul.verma9@gmail.com -> r***@gmail.com)
CREATE OR REPLACE FUNCTION fk_1368.gold.mask_email(email STRING)
RETURNS STRING
RETURN CASE
  WHEN is_member('fk_admins') THEN email
  WHEN EXISTS (
    SELECT 1 FROM fk_1368.ops.user_entitlements
    WHERE user_email = current_user() AND can_view_pii
  ) THEN email
  ELSE concat(substr(email, 1, 1), '***@', substr(email, instr(email, '@') + 1))
END;

-- Phone mask: XXXXXX + last 4 digits (e.g., 9876543210 -> XXXXXX3210)
CREATE OR REPLACE FUNCTION fk_1368.gold.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE
  WHEN is_member('fk_admins') THEN phone
  WHEN EXISTS (
    SELECT 1 FROM fk_1368.ops.user_entitlements
    WHERE user_email = current_user() AND can_view_pii
  ) THEN phone
  ELSE concat('XXXXXX', substr(phone, -4))
END;

-- Apply masks to dim_customer
ALTER TABLE fk_1368.gold.dim_customer
ALTER COLUMN email SET MASK fk_1368.gold.mask_email;

ALTER TABLE fk_1368.gold.dim_customer
ALTER COLUMN phone SET MASK fk_1368.gold.mask_phone;